# musicgen demo

End-to-end feature showcase for [musicgen](https://github.com/dobidu/musicgen) — a synthetic music dataset generator.

**Requires:** Python ≥ 3.10, FluidSynth binary, at least one `.sf2` per layer in `sf/<layer>/`.

Cells tagged `# requires: fluidsynth` are skipped gracefully when the binary is absent.

## 1. Setup

In [ ]:
import shutil, sys, os

# Add repo root to sys.path so config.py is importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config import Config
from musicgen import __version__

HAS_FLUIDSYNTH = shutil.which("fluidsynth") is not None
print(f"musicgen {__version__}")
print(f"FluidSynth present: {HAS_FLUIDSYNTH}")

cfg = Config()
print(f"dataset_root: {cfg.dataset_root}")
print(f"sf_dir:       {cfg.sf_dir}")

## 2. Single sample generation

In [ ]:
# requires: fluidsynth
import json, tempfile, pathlib
from musicgen import generate

if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    with tempfile.TemporaryDirectory(prefix="musicgen_demo_") as tmpdir:
        cfg_single = Config.load(cli_overrides={
            "global_seed": 42,
            "sample_index": 0,
            "dataset_root": tmpdir,
        })
        result = generate(cfg_single)
        print(f"status:            {result.status}")
        print(f"seed:              {result.seed}")
        print(f"split:             {result.split}")
        print(f"musicality_score:  {result.musicality_score:.3f}")
        print(f"duration_seconds:  {result.duration_seconds:.2f} s")
        
        sample_json = pathlib.Path(result.sample_json_path)
        if sample_json.exists():
            data = json.loads(sample_json.read_text())
            print("\nsample.json fields:", list(data.keys()))

## 3. Batch generation

In [ ]:
# requires: fluidsynth
import tempfile
from musicgen.batch import generate_batch

if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    with tempfile.TemporaryDirectory(prefix="musicgen_batch_") as tmpdir:
        cfg_batch = Config.load(cli_overrides={
            "global_seed": 100,
            "count": 3,
            "dataset_root": tmpdir,
            "workers": 2,
        })
        batch_result = generate_batch(cfg_batch)
        print(f"Succeeded: {batch_result.succeeded}")
        print(f"Failed:    {batch_result.failed}")
        print(f"Total:     {batch_result.total}")
        print(f"Duration:  {batch_result.duration_seconds:.1f} s")

## 4. Output modes

In [ ]:
# requires: fluidsynth
import tempfile
from musicgen import generate

MODES = ["full", "stems-only", "midi-only"]

if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    for mode in MODES:
        with tempfile.TemporaryDirectory() as tmpdir:
            cfg_mode = Config.load(cli_overrides={
                "global_seed": 7,
                "sample_index": 0,
                "dataset_root": tmpdir,
                "output_mode": mode,
            })
            r = generate(cfg_mode)
            stems = list(r.stem_paths.values())
            midis = list(r.midi_paths.values())
            print(f"mode={mode:<12} mix={'yes' if r.mix_path else 'no':3} "
                  f"stems={len(stems)} midis={len(midis)}")

## 5. Genre generation — single genre

In [ ]:
# requires: fluidsynth
import json, tempfile, pathlib
from musicgen import generate
from musicgen.genre import load_genre

if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    jazz_spec = load_genre("jazz", os.path.join(REPO_ROOT, "genres"))
    print(f"jazz tempo range: [{jazz_spec.tempo_min}, {jazz_spec.tempo_max}] BPM")
    print(f"jazz swing range: [{jazz_spec.swing_min}, {jazz_spec.swing_max}]")
    print(f"jazz time_sig_weights: {jazz_spec.time_sig_weights}")

    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_jazz = Config.load(cli_overrides={
            "global_seed": 42,
            "sample_index": 0,
            "dataset_root": tmpdir,
            "genre": ["jazz"],
        })
        r = generate(cfg_jazz)
        sample_data = json.loads(pathlib.Path(r.sample_json_path).read_text())
        tempo = sample_data.get("tempo_bpm")
        swing = sample_data.get("swing")
        print(f"\nGenerated: tempo={tempo} BPM, swing={swing:.3f}")
        assert jazz_spec.tempo_min <= tempo <= jazz_spec.tempo_max, "tempo out of jazz range!"
        print("Tempo within jazz bounds: OK")


## 6. Genre composition — jazz + latin

In [ ]:
from musicgen.genre import load_genre, merge_genres

genres_dir = os.path.join(REPO_ROOT, "genres")
jazz = load_genre("jazz", genres_dir)
latin = load_genre("latin", genres_dir)

merged = merge_genres([jazz, latin])

print("=== Genre composition: jazz + latin ===")
print(f"jazz  tempo: [{jazz.tempo_min}, {jazz.tempo_max}]")
print(f"latin tempo: [{latin.tempo_min}, {latin.tempo_max}]")
print(f"merged tempo (intersection): [{merged.tempo_min}, {merged.tempo_max}]")
print()
print(f"jazz  swing: [{jazz.swing_min}, {jazz.swing_max}]")
print(f"latin swing: [{latin.swing_min}, {latin.swing_max}]")
print(f"merged swing (intersection): [{merged.swing_min}, {merged.swing_max}]")
print()
print(f"merged time_sig_weights: {merged.time_sig_weights}")

## 7. Audio playback

In [ ]:
# requires: fluidsynth + IPython.display
import tempfile
from musicgen import generate

try:
    from IPython.display import Audio, display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

if not HAS_FLUIDSYNTH or not HAS_IPYTHON:
    print("Skipping: requires FluidSynth + IPython")
else:
    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_play = Config.load(cli_overrides={
            "global_seed": 5,
            "sample_index": 0,
            "dataset_root": tmpdir,
        })
        r = generate(cfg_play)
        if r.mix_path:
            print("Mix audio:")
            display(Audio(r.mix_path))
        for layer, stem_path in r.stem_paths.items():
            print(f"Stem — {layer}:")
            display(Audio(stem_path))

## 8. MIDI visualization

In [ ]:
# requires: fluidsynth + pretty_midi + matplotlib
import tempfile
from musicgen import generate

try:
    import pretty_midi
    import matplotlib.pyplot as plt
    HAS_PRETTY_MIDI = True
except ImportError:
    HAS_PRETTY_MIDI = False

if not HAS_FLUIDSYNTH or not HAS_PRETTY_MIDI:
    print("Skipping: requires FluidSynth + pretty_midi + matplotlib")
else:
    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_midi = Config.load(cli_overrides={
            "global_seed": 3,
            "sample_index": 0,
            "dataset_root": tmpdir,
        })
        r = generate(cfg_midi)

        # r.midi_paths is flat: {layer: path}
        for layer, midi_path in r.midi_paths.items():
            if layer == "beat":
                continue  # drums less informative in piano roll
            pm = pretty_midi.PrettyMIDI(midi_path)
            roll = pm.get_piano_roll(fs=10)
            fig, ax = plt.subplots(figsize=(12, 3))
            ax.imshow(roll, aspect="auto", origin="lower", cmap="Blues")
            ax.set_title(f"layer: {layer}")
            ax.set_xlabel("Time")
            ax.set_ylabel("MIDI note")
            plt.tight_layout()
            plt.show()


## 9. Musicality score

In [ ]:
# requires: fluidsynth
import tempfile
from musicgen import generate
from musicgen.batch import generate_batch

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    N = 4
    scores = []
    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_scores = Config.load(cli_overrides={
            "global_seed": 200,
            "count": N,
            "dataset_root": tmpdir,
        })
        batch_result = generate_batch(cfg_scores)
        # Re-read from sample.json
        import json, pathlib
        for i in range(N):
            sj = pathlib.Path(tmpdir) / f"{i:06d}" / "sample.json"
            if sj.exists():
                d = json.loads(sj.read_text())
                ms = d.get("musicality_score", {})
                scores.append({"index": i, "score": ms.get("score", 0.0)})

    if HAS_MPL and scores:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.bar([s["index"] for s in scores], [s["score"] for s in scores])
        ax.set_xlabel("Sample index")
        ax.set_ylabel("Musicality score")
        ax.set_title(f"Musicality scores ({N} samples, seed=200)")
        plt.tight_layout()
        plt.show()
    else:
        for s in scores:
            print(f"sample {s['index']}: {s['score']:.3f}")


## 10. MIDI indexing

In [ ]:
# requires: fluidsynth + midi_file_manager
import tempfile
from musicgen import generate

try:
    import midi_manager  # the external dep called by midi_indexer
    from musicgen.midi_indexer import index_midi_dataset
    HAS_MIDI_IDX = True
except ImportError:
    HAS_MIDI_IDX = False

if not HAS_FLUIDSYNTH or not HAS_MIDI_IDX:
    print("Skipping: requires FluidSynth + midi_file_manager")
else:
    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_idx = Config.load(cli_overrides={
            "global_seed": 300,
            "sample_index": 0,
            "dataset_root": tmpdir,
        })
        generate(cfg_idx)

        db_path = os.path.join(tmpdir, "midi_db.json")
        count = index_midi_dataset(dataset_root=tmpdir, out_db=db_path)
        print(f"Indexed {count} MIDI entries → {db_path}")


## 11. Audio indexing

In [ ]:
# requires: fluidsynth + audio_sample_manager
import tempfile
from musicgen import generate

try:
    import audio_sample_manager  # the external dep called by audio_indexer
    from musicgen.audio_indexer import index_audio_dataset
    HAS_AUDIO_IDX = True
except ImportError:
    HAS_AUDIO_IDX = False

if not HAS_FLUIDSYNTH or not HAS_AUDIO_IDX:
    print("Skipping: requires FluidSynth + audio_sample_manager")
else:
    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_aidx = Config.load(cli_overrides={
            "global_seed": 400,
            "sample_index": 0,
            "dataset_root": tmpdir,
        })
        generate(cfg_aidx)

        db_path = os.path.join(tmpdir, "audio_db.json")
        count = index_audio_dataset(dataset_root=tmpdir, out_db=db_path)
        print(f"Indexed {count} audio entries → {db_path}")


## 12. Determinism check

In [ ]:
# requires: fluidsynth
import hashlib, json, tempfile, pathlib
from musicgen import generate

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

if not HAS_FLUIDSYNTH:
    print("Skipping: FluidSynth not found")
else:
    SEED, INDEX = 42, 0
    hashes = []
    for run in range(2):
        with tempfile.TemporaryDirectory() as tmpdir:
            cfg_det = Config.load(cli_overrides={
                "global_seed": SEED,
                "sample_index": INDEX,
                "dataset_root": tmpdir,
            })
            r = generate(cfg_det)
            sj = pathlib.Path(r.sample_json_path)
            hashes.append(sha256_file(sj))
    
    match = hashes[0] == hashes[1]
    print(f"Run 1 SHA-256: {hashes[0][:16]}...")
    print(f"Run 2 SHA-256: {hashes[1][:16]}...")
    print(f"Determinism: {'PASS' if match else 'FAIL'}")
    assert match, "Determinism broken!"

## 13. Advanced workflows

This notebook covers core musicgen features (v0.1–v0.3). For more advanced workflows:

- **`sample_composition.ipynb`** — mix real audio samples alongside / instead of FluidSynth-rendered layers (v0.4, requires `musicgen[samples]`).
- **`neural_generators.ipynb`** — train and use LSTM-based chord / melody generators (v0.5, requires `musicgen[neural]`).

Both notebooks live in this `notebooks/` directory.
